In [3]:
from pathlib import Path
import pandas as pd

file_path = Path(
    r"C:\Users\hp\Downloads\archive (3)"
    r"\healthcare_appointment_no_show_wait_time.csv"
)

no_show_data = pd.read_csv(file_path)

print("Loaded records:", len(no_show_data))

Loaded records: 2800


In [5]:
no_show_model_data = no_show_data[
    [
        "department",
        "appointment_type",
        "scheduled_hour",
        "reminder_sent",
        "previous_no_shows",
        "appointment_status",
    ]
].copy()

no_show_model_data["is_no_show"] = (
    no_show_model_data["appointment_status"]
    .eq("No-Show")
    .astype(int)
)

no_show_model_data = no_show_model_data.drop(
    columns="appointment_status"
)

X_no_show = no_show_model_data.drop(
    columns="is_no_show"
)

y_no_show = no_show_model_data["is_no_show"]

print("No-show rate:", round(y_no_show.mean() * 100, 2), "%")
print(y_no_show.value_counts())

No-show rate: 23.07 %
is_no_show
0    2154
1     646
Name: count, dtype: int64


In [7]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

X_train_ns, X_test_ns, y_train_ns, y_test_ns = (
    train_test_split(
        X_no_show,
        y_no_show,
        test_size=0.20,
        random_state=42,
        stratify=y_no_show,
    )
)

baseline_classifier = DummyClassifier(
    strategy="most_frequent"
)

baseline_classifier.fit(X_train_ns, y_train_ns)

baseline_predictions = baseline_classifier.predict(
    X_test_ns
)

baseline_probabilities = (
    baseline_classifier.predict_proba(X_test_ns)[:, 1]
)

print(
    "Baseline accuracy:",
    round(
        accuracy_score(
            y_test_ns,
            baseline_predictions,
        ),
        4,
    ),
)

print(
    "Baseline balanced accuracy:",
    round(
        balanced_accuracy_score(
            y_test_ns,
            baseline_predictions,
        ),
        4,
    ),
)

print(
    "Baseline ROC-AUC:",
    round(
        roc_auc_score(
            y_test_ns,
            baseline_probabilities,
        ),
        4,
    ),
)

print("\nClassification report:")
print(
    classification_report(
        y_test_ns,
        baseline_predictions,
        target_names=[
            "Attended",
            "No-show",
        ],
        zero_division=0,
    )
)

Baseline accuracy: 0.7696
Baseline balanced accuracy: 0.5
Baseline ROC-AUC: 0.5

Classification report:
              precision    recall  f1-score   support

    Attended       0.77      1.00      0.87       431
     No-show       0.00      0.00      0.00       129

    accuracy                           0.77       560
   macro avg       0.38      0.50      0.43       560
weighted avg       0.59      0.77      0.67       560



In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features_ns = [
    "scheduled_hour",
    "previous_no_shows",
]

categorical_features_ns = [
    "department",
    "appointment_type",
    "reminder_sent",
]

preprocessor_ns = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features_ns,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent",
                        ),
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                        ),
                    ),
                ],
            ),
            categorical_features_ns,
        ),
    ],
)

no_show_classifier = Pipeline(
    steps=[
        ("preprocessor", preprocessor_ns),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=10,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ],
)

no_show_classifier.fit(X_train_ns, y_train_ns)

ns_predictions = no_show_classifier.predict(X_test_ns)
ns_probabilities = no_show_classifier.predict_proba(
    X_test_ns
)[:, 1]

print(
    "Accuracy:",
    round(accuracy_score(y_test_ns, ns_predictions), 4),
)

print(
    "Balanced accuracy:",
    round(
        balanced_accuracy_score(y_test_ns, ns_predictions),
        4,
    ),
)

print(
    "ROC-AUC:",
    round(
        roc_auc_score(y_test_ns, ns_probabilities),
        4,
    ),
)

print("\nClassification report:")
print(
    classification_report(
        y_test_ns,
        ns_predictions,
        target_names=["Attended", "No-show"],
        zero_division=0,
    )
)

Accuracy: 0.5321
Balanced accuracy: 0.4978
ROC-AUC: 0.4739

Classification report:
              precision    recall  f1-score   support

    Attended       0.77      0.56      0.65       431
     No-show       0.23      0.43      0.30       129

    accuracy                           0.53       560
   macro avg       0.50      0.50      0.47       560
weighted avg       0.64      0.53      0.57       560



## Conclusion: No-Show Classification

- The dataset contained 2,800 appointments with a 23.07% no-show rate.
- A majority-class baseline achieved 76.96% accuracy but identified no no-show appointments.
- Random Forest with balanced class weights achieved ROC-AUC 0.4739, below random performance.
- The available fields did not provide reliable no-show prediction signal.
- This classifier is rejected and will not be saved, deployed, or used for patient booking decisions.
- SmartCare should use no-show information only after collecting validated, hospital-governed operational data and performing fairness review.